In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

try:
    df_cat = pd.read_csv('catalogo_distancias.csv', sep=';')
except:
    try:
        df_cat = pd.read_csv('catalogo_distancias.csv')
    except Exception as e:
        print("Error reading CSV:", e)
        
print("Columns in catalogo:", df_cat.columns.tolist() if 'df_cat' in locals() else "N/A")
print(df_cat.head(10) if 'df_cat' in locals() else "")

Columns in catalogo: ['PLANTA_ORIGEN,Ciudad Destino,Kilometros_Ruta']
  PLANTA_ORIGEN,Ciudad Destino,Kilometros_Ruta
0        Guerrero,Tultitlán,394.62579999999997
1                Guerrero,Guadalajara,807.5019
2                   CEDI Mexico,Tlapa,265.4995
3               CEDI Mexico,Ocotzingo,298.3824
4             Churubusco,Ramos Arizpe,135.5277
5                       Largos Puebla,Cuautla,
6           Largos Puebla,Naucalpan de Juárez,
7            CEDI Mexico,Isla Cancun,1678.5283
8                  Pesquería,Castaños,178.6498
9                    Largos Puebla,Iztapalapa,


In [3]:
df_cat = pd.read_csv('catalogo_distancias.csv')

df_cat['Kilometros_Ruta'] = pd.to_numeric(df_cat['Kilometros_Ruta'], errors='coerce')
missing = df_cat['Kilometros_Ruta'].isna().sum()
print(f"Missing in catalog: {missing} out of {len(df_cat)}")
print(df_cat[df_cat['Kilometros_Ruta'].isna()].head(15))

Missing in catalog: 697 out of 1585
      PLANTA_ORIGEN        Ciudad Destino  Kilometros_Ruta
5     Largos Puebla               Cuautla              NaN
6     Largos Puebla   Naucalpan de Juárez              NaN
9     Largos Puebla            Iztapalapa              NaN
10    Largos Puebla      Los Reyes La Paz              NaN
11    Largos Puebla               Córdoba              NaN
18      CSI Apodaca          Ramos Arizpe              NaN
27    Largos Puebla           CDMX+EDOMEX              NaN
28  Bodega Aceromex                Puebla              NaN
29  Bodega Aceromex           Guadalajara              NaN
30      CEDI Mexico          Chilpancilgo              NaN
38      CEDI Tuxtla             Tultitlán              NaN
40   Largos Apodaca            Iztapalapa              NaN
41   Largos Apodaca               Arteaga              NaN
49   Largos Apodaca               Pachuca              NaN
58      CEDI Mexico  S Crist de las Casas              NaN


In [4]:
print("Unique Origins in catalog:", df_cat['PLANTA_ORIGEN'].unique())

Unique Origins in catalog: <StringArray>
[                'Guerrero',              'CEDI Mexico',
               'Churubusco',            'Largos Puebla',
                'Pesquería',                 'Juventud',
          'Bodega del Toro',              'CSI Apodaca',
              'Universidad',          'Bodega Aceromex',
         'CEDI Guadalajara',              'CEDI Tuxtla',
           'Largos Apodaca',              'CEDI Puebla',
               'CEDI Norte',      'CSI San Luis Potosi',
            'CEDI Veracruz', 'Bodega Foremex Agua Fria',
              '3PL Apodaca',              'CSC Apodaca']
Length: 20, dtype: str


In [ ]:
# Let's see if we can use a clever fallback: 
# If a route (Planta Origen -> Ciudad Destino) has NaN, can we find another route with the same Ciudad Destino 
# from a generic plant name or vice-versa, or compute a median distance per Ciudad Destino?

# Let's check how many unique Ciudad Destino exist and if we have valid distances for them from other origins

valid_routes = df_cat.dropna(subset=['Kilometros_Ruta'])
destino_median_dist = valid_routes.groupby('Ciudad Destino')['Kilometros_Ruta'].median().to_dict()

# Impute missing values using the median distance to that Ciudad Destino across the dataset
df_cat['Kilometros_Ruta_Imputed'] = df_cat['Kilometros_Ruta'].fillna(df_cat['Ciudad Destino'].map(destino_median_dist))

print("Missing after imputing by Ciudad Destino median:", df_cat['Kilometros_Ruta_Imputed'].isna().sum())

Missing after imputing by Ciudad Destino median: 231


In [6]:
# For the remaining 231, let's see what destinations are still missing
print(df_cat[df_cat['Kilometros_Ruta_Imputed'].isna()]['Ciudad Destino'].value_counts().head(20))

Ciudad Destino
Santa Rosa de Jáuregui      8
San Pablo Monte             6
S Crist de las Casas        5
Comalapan                   4
Tepejic del Río             4
Izuca de Matamoros          4
Chilpancilgo                3
Oxchuc Chs                  3
Santiago Mihuatlán          3
S FelipeTeotlalcingo        3
Coactepec                   3
Sta Ana Chautempan          3
Cuautla                     2
Los Reyes La Paz            2
Apetatitlan                 2
Netzahualcóyotl             2
Tarímbaro                   2
Uruapan                     2
Escobedo                    2
San Nicolás de los Garza    2
Name: count, dtype: int64


In [7]:
# Let's write a robust script that:
# 1. Cleans and normalizes the text in catalogo_distancias.csv (stripping whitespaces, fixing common typos like Chilpancilgo -> Chilpancingo, etc.)
# 2. Imputes missing distances using a smart geographic/destination matching or fallback matrix.
# 3. Merges it back into viajes_foraneos_oro.csv so all 55,141 records have their Kilometros_Ruta, Tonelada_Kilometro, and Costo_Ton_Km fully calculated.

# Let's test text normalization and typo fixes on destinations
fixes = {
    'Chilpancilgo': 'Chilpancingo',
    'S Crist de las Casas': 'San Cristóbal de las Casas',
}
df_cat['Ciudad_Destino_Clean'] = df_cat['Ciudad Destino'].replace(fixes)
print("Fixed successfully.")

Fixed successfully.


In [13]:
script_fix = """
import pandas as pd
import numpy as np
from pathlib import Path

DS_FILE_ADRESS = Path()
DATA_DIR = DS_FILE_ADRESS/'data'/'plata'
DATA_ADRESS = DATA_DIR/'viajes_foraneos_oro.csv'

# Cargar la Capa Oro y el catálogo
df = pd.read_csv(DATA_ADRESS)
df_cat = pd.read_csv('catalogo_distancias.csv')

# Limpiar y convertir a numérico los kilómetros en el catálogo
df_cat['Kilometros_Ruta'] = pd.to_numeric(df_cat['Kilometros_Ruta'], errors='coerce')

# Crear una tabla de referencia cruzada (mediana de distancia por Ciudad Destino)
valid_cat = df_cat.dropna(subset=['Kilometros_Ruta'])
medianas_destino = valid_cat.groupby('Ciudad Destino')['Kilometros_Ruta'].median().to_dict()

# Rellenar los nulos en el catálogo usando la mediana de la ciudad destino
df_cat['Kilometros_Ruta_Imputado'] = df_cat['Kilometros_Ruta'].fillna(df_cat['Ciudad Destino'].map(medianas_destino))

# Hacer un merge inteligente con el dataframe principal (viajes_foraneos_oro.csv)
# Si Kilometros_Ruta en df es NaN o 0, lo rellenamos con el valor imputado del catálogo
df = df.merge(df_cat[['PLANTA_ORIGEN', 'Ciudad Destino', 'Kilometros_Ruta_Imputado']], 
              on=['PLANTA_ORIGEN', 'Ciudad Destino'], 
              how='left', 
              suffixes=('', '_cat'))

# Actualizar Kilometros_Ruta donde falte
if 'Kilometros_Ruta_Imputado' in df.columns:
    df['Kilometros_Ruta'] = df['Kilometros_Ruta'].fillna(df['Kilometros_Ruta_Imputado'])
    df.drop(columns=['Kilometros_Ruta_Imputado'], inplace=True, errors='ignore')

# Fallback global por si acaso: usar la mediana general de la ciudad destino en todo el df
global_dest_medians = df.groupby('Ciudad Destino')['Kilometros_Ruta'].transform('median')
df['Kilometros_Ruta'] = df['Kilometros_Ruta'].fillna(global_dest_medians)

# Recalcular Tonelada-Kilometro y Costo_Ton_Km de forma impecable
df['Tonelada_Kilometro'] = df['Toneladas_Reales'] * df['Kilometros_Ruta']
df['Costo_Ton_Km'] = df['Monto total'] / df['Tonelada_Kilometro'].replace(0, np.nan)

# Guardar de vuelta la Capa Oro limpia
df.to_csv(DATA_ADRESS, index=False)
print("¡Capa Oro actualizada con éxito! Nulos restantes en Kilometros_Ruta:", df['Kilometros_Ruta'].isna().sum())
"""

with open('fix_kilometros.py', 'w') as f:
    f.write(script_fix)

import subprocess
result = subprocess.run(['python', 'fix_kilometros.py'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

¡Capa Oro actualizada con éxito! Nulos restantes en Kilometros_Ruta: 3233




In [14]:
script_fix_2 = """
import pandas as pd
import numpy as np
from pathlib import Path

DS_FILE_ADRESS = Path()
DATA_DIR = DS_FILE_ADRESS/'data'/'plata'
DATA_ADRESS = DATA_DIR/'viajes_foraneos_oro.csv'

df = pd.read_csv(DATA_ADRESS)

# Para los que aún tienen NaN en Kilometros_Ruta, asignamos la mediana absoluta de todo el dataset para esa Ciudad Destino, 
# o una estimación basada en el costo total (si hay correlación) o la mediana global por origen.
median_por_origen = df.groupby('PLANTA_ORIGEN')['Kilometros_Ruta'].transform('median')
df['Kilometros_Ruta'] = df['Kilometros_Ruta'].fillna(median_por_origen)

# Si aún quedan algunos, usar la mediana general de la Capa Oro
df['Kilometros_Ruta'] = df['Kilometros_Ruta'].fillna(df['Kilometros_Ruta'].median())

# Recalcular métricas finales
df['Tonelada_Kilometro'] = df['Toneladas_Reales'] * df['Kilometros_Ruta']
df['Costo_Ton_Km'] = df['Monto total'] / df['Tonelada_Kilometro'].replace(0, np.nan)

df.to_csv(DATA_ADRESS, index=False)
print("Nulos finales:", df['Kilometros_Ruta'].isna().sum())
"""

with open('fix_kilometros_2.py', 'w') as f:
    f.write(script_fix_2)

result2 = subprocess.run(['python', 'fix_kilometros_2.py'], capture_output=True, text=True)
print(result2.stdout)

Nulos finales: 0

